# Scheduling Agent Work with the CueAPI MCP Server

This cookbook shows how to give Claude the ability to **schedule its own future work** and **prove that work actually happened**, using the open-source [CueAPI](https://cueapi.ai) Model Context Protocol (MCP) server.

- MCP registry listing: [`ai.cueapi/mcp`](https://registry.modelcontextprotocol.io/v0/servers?search=ai.cueapi)
- GitHub: [cueapi/cueapi-mcp](https://github.com/cueapi/cueapi-mcp)
- npm: [`@cueapi/mcp`](https://www.npmjs.com/package/@cueapi/mcp)

> **Community contribution.** CueAPI is not affiliated with Anthropic. This notebook demonstrates a third-party MCP server that any Claude client (Claude Desktop, Claude Code, or the Messages API) can connect to.

## What this cookbook demonstrates

1. **Installing and configuring** the `@cueapi/mcp` server for Claude Desktop.
2. **Example 1 — Scheduling**: ask Claude to create a daily cue at 9am. Claude calls the `cueapi_create_cue` tool.
3. **Example 2 — History**: ask Claude what happened on past runs. Claude calls `cueapi_list_executions`.
4. **Example 3 — Verified outcomes**: after the agent does its work, it reports back to CueAPI with evidence (URL, external ID, summary). This is the *accountability primitive* — an audit trail that says 'the agent claimed this, here is the proof'.
5. **Why this matters**: a closing note on the failure mode where agents *say* they did the work but didn't — and how verified outcomes make that lie expensive.

## How the cells work

The code cells below use the Anthropic Messages API directly, with **tool definitions that mirror exactly what the CueAPI MCP server exposes to Claude Desktop**. This keeps the cookbook runnable from a notebook (no Claude Desktop process required) while faithfully modelling the MCP tool contract.

If you prefer to run this end-to-end through Claude Desktop, skip to [Step 1](#step-1-install-and-configure-cueapi-mcp) for the config file and then come back to the tool definitions to see what Claude sees on the other side.

## Step 1: Install and configure `@cueapi/mcp`

The CueAPI MCP server is a stdio-transport Node.js package. Install it globally or via `npx`:

```bash
npm install -g @cueapi/mcp
```

Sign up at [cueapi.ai](https://cueapi.ai) and grab an API key, then add the server to your `claude_desktop_config.json`:

```json
{
  "mcpServers": {
    "cueapi": {
      "command": "npx",
      "args": ["-y", "@cueapi/mcp"],
      "env": {
        "CUEAPI_API_KEY": "cue_sk_..."
      }
    }
  }
}
```

Restart Claude Desktop. You should see 8 CueAPI tools appear in the MCP tool picker:
`cueapi_create_cue`, `cueapi_list_cues`, `cueapi_get_cue`, `cueapi_pause_cue`, `cueapi_resume_cue`, `cueapi_delete_cue`, `cueapi_list_executions`, `cueapi_report_outcome`.

## Step 2: Set up the notebook environment

Install the Anthropic Python SDK and configure a client. The cells below talk to the Messages API directly and hand-craft `tool_use` / `tool_result` blocks to match the MCP contract — useful both for testing your MCP integration and for embedding the same tool surface in a Messages-API-driven agent loop.

In [ ]:
%pip install anthropic

In [ ]:
import json, os
from anthropic import Anthropic

client = Anthropic()
MODEL_NAME = "claude-sonnet-4-5"

# The CueAPI MCP server runs as a separate process when using Claude Desktop.
# When driving from a notebook we mock the handler so the notebook is fully
# self-contained — in production, Claude Desktop routes these tool calls to
# the @cueapi/mcp process, which forwards to https://api.cueapi.ai.
CUEAPI_API_KEY = os.environ.get("CUEAPI_API_KEY", "cue_sk_demo")
print("Anthropic client ready. Model:", MODEL_NAME)

Anthropic client ready. Model: claude-sonnet-4-5


## Step 3: Define the CueAPI tool surface

These are the exact tool schemas `@cueapi/mcp` exposes. Each one maps 1:1 to a CueAPI REST endpoint. When Claude Desktop connects to the MCP server, these are the tools it sees; when we call the Messages API below, we pass them inline.

In [ ]:
CUEAPI_TOOLS = [
    {
        "name": "cueapi_create_cue",
        "description": (
            "Create a new CueAPI cue — a scheduled job that fires a callback "
            "(or enqueues worker work) on a cron or one-time trigger."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "name": {"type": "string", "description": "Human-readable cue name"},
                "cron": {"type": "string", "description": "Cron expression for a recurring cue (e.g. '0 9 * * *')"},
                "at": {"type": "string", "description": "ISO-8601 timestamp for a one-time cue"},
                "callback_url": {"type": "string", "description": "Webhook URL fired when the cue triggers"},
                "worker": {"type": "boolean", "description": "If true, use worker transport — no callback URL needed"},
                "timezone": {"type": "string", "description": "IANA timezone, default 'UTC'"},
                "payload": {"type": "object", "description": "Arbitrary JSON payload delivered with the cue"},
                "description": {"type": "string"},
            },
            "required": ["name"],
        },
    },
    {
        "name": "cueapi_list_executions",
        "description": (
            "List executions — the historical record of times a cue actually fired. "
            "Optionally filter by cue, status, or paginate."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "cue_id": {"type": "string", "description": "Filter to a specific cue"},
                "status": {"type": "string", "description": "Filter by execution status"},
                "limit": {"type": "integer"},
                "offset": {"type": "integer"},
            },
        },
    },
    {
        "name": "cueapi_report_outcome",
        "description": (
            "Report the outcome of an execution. CueAPI's core accountability primitive: "
            "attach evidence (external_id, result_url, summary) that proves the work actually "
            "happened. Write-once — the outcome record is immutable."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "execution_id": {"type": "string"},
                "success": {"type": "boolean"},
                "external_id": {"type": "string", "description": "ID from the downstream system"},
                "result_url": {"type": "string", "description": "Public URL proving the work happened (tweet, PR, etc.)"},
                "summary": {"type": "string", "description": "Short human summary of what the agent did"},
            },
            "required": ["execution_id", "success"],
        },
    },
]

# We focus on three tools for this cookbook. The full server exposes eight —
# see https://github.com/cueapi/cueapi-mcp/blob/main/src/tools.ts for the rest
# (list_cues, get_cue, pause_cue, resume_cue, delete_cue).
print(f"Loaded {len(CUEAPI_TOOLS)} CueAPI tool definitions.")

Loaded 3 CueAPI tool definitions.


## Step 4: A minimal MCP-style tool runner

In Claude Desktop, `@cueapi/mcp` receives tool calls over stdio and forwards them to the CueAPI API. For this notebook we stand in a mock that returns realistic responses, so you can see the full agent loop without needing live credentials. Swap `mock_cueapi_handler` for `real_cueapi_handler` (below) to run against your account.

In [ ]:
from typing import Any

# Deterministic mock state so the notebook renders identically each run.
_MOCK_STATE = {
    "cues": {},
    "executions": {},
    "outcomes": {},
}

def mock_cueapi_handler(tool_name: str, tool_input: dict[str, Any]) -> dict[str, Any]:
    """Stand-in for the @cueapi/mcp -> https://api.cueapi.ai round trip."""
    if tool_name == "cueapi_create_cue":
        cue_id = f"cue_{len(_MOCK_STATE['cues']) + 1:04d}"
        cue = {
            "id": cue_id,
            "name": tool_input["name"],
            "cron": tool_input.get("cron"),
            "callback_url": tool_input.get("callback_url"),
            "worker": tool_input.get("worker", False),
            "timezone": tool_input.get("timezone", "UTC"),
            "next_fire_at": "2026-04-16T09:00:00Z",
            "status": "active",
            "created_at": "2026-04-15T21:30:00Z",
        }
        _MOCK_STATE["cues"][cue_id] = cue
        # Seed some history for this cue so Example 2 has data to show.
        for i, (iso, status) in enumerate([
            ("2026-04-13T09:00:00Z", "succeeded"),
            ("2026-04-14T09:00:00Z", "succeeded"),
            ("2026-04-15T09:00:00Z", "succeeded"),
        ]):
            exec_id = f"exe_{cue_id}_{i+1:03d}"
            _MOCK_STATE["executions"][exec_id] = {
                "id": exec_id,
                "cue_id": cue_id,
                "fired_at": iso,
                "status": status,
                "duration_ms": 842 + i * 40,
                "outcome": _MOCK_STATE["outcomes"].get(exec_id),
            }
        return cue
    if tool_name == "cueapi_list_executions":
        cue_id = tool_input.get("cue_id")
        items = [
            e for e in _MOCK_STATE["executions"].values()
            if cue_id is None or e["cue_id"] == cue_id
        ]
        items.sort(key=lambda e: e["fired_at"], reverse=True)
        return {"executions": items, "total": len(items)}
    if tool_name == "cueapi_report_outcome":
        exec_id = tool_input["execution_id"]
        outcome = {
            "execution_id": exec_id,
            "success": tool_input["success"],
            "external_id": tool_input.get("external_id"),
            "result_url": tool_input.get("result_url"),
            "summary": tool_input.get("summary"),
            "reported_at": "2026-04-15T21:31:12Z",
            "immutable": True,
        }
        _MOCK_STATE["outcomes"][exec_id] = outcome
        if exec_id in _MOCK_STATE["executions"]:
            _MOCK_STATE["executions"][exec_id]["outcome"] = outcome
        return outcome
    return {"error": f"unknown tool: {tool_name}"}

def real_cueapi_handler(tool_name: str, tool_input: dict[str, Any]) -> dict[str, Any]:
    """Swap in to hit the real CueAPI API. Requires CUEAPI_API_KEY."""
    import urllib.request, urllib.error
    from urllib.parse import urlencode
    method, path, body = _cueapi_route(tool_name, tool_input)
    url = f"https://api.cueapi.ai{path}"
    if method == "GET" and tool_input:
        qs = urlencode({k: v for k, v in tool_input.items() if v is not None})
        if qs:
            url = f"{url}?{qs}"
    req = urllib.request.Request(
        url, method=method,
        headers={
            "Authorization": f"Bearer {CUEAPI_API_KEY}",
            "Content-Type": "application/json",
        },
        data=json.dumps(body).encode() if body else None,
    )
    with urllib.request.urlopen(req, timeout=10) as resp:
        return json.loads(resp.read().decode())

def _cueapi_route(tool_name, args):
    if tool_name == "cueapi_create_cue":
        return "POST", "/v1/cues", {k: v for k, v in args.items() if v is not None}
    if tool_name == "cueapi_list_executions":
        return "GET", "/v1/executions", None
    if tool_name == "cueapi_report_outcome":
        exec_id = args["execution_id"]
        body = {k: v for k, v in args.items() if k != "execution_id" and v is not None}
        return "POST", f"/v1/executions/{exec_id}/outcome", body
    raise ValueError(f"unknown tool: {tool_name}")

cueapi_handler = mock_cueapi_handler
print("Tool handler wired. Using: mock_cueapi_handler")

Tool handler wired. Using: mock_cueapi_handler


In [ ]:
def run_agent_turn(user_message: str, max_turns: int = 4) -> list[dict]:
    """Minimal agent loop: send user message, resolve tool calls, return transcript."""
    messages = [{"role": "user", "content": user_message}]
    transcript = []
    for turn in range(max_turns):
        resp = client.messages.create(
            model=MODEL_NAME,
            max_tokens=1024,
            tools=CUEAPI_TOOLS,
            messages=messages,
        )
        transcript.append({"role": "assistant", "content": resp.content})
        if resp.stop_reason != "tool_use":
            break
        tool_results = []
        for block in resp.content:
            if block.type == "tool_use":
                result = cueapi_handler(block.name, block.input)
                tool_results.append({
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": json.dumps(result),
                })
        messages.append({"role": "assistant", "content": resp.content})
        messages.append({"role": "user", "content": tool_results})
        transcript.append({"role": "tool_result", "content": tool_results})
    return transcript

## Example 1 — "Schedule a daily check at 9am"

The user wants Claude to schedule a recurring job that pings a health endpoint every morning. Claude should recognise this maps to `cueapi_create_cue` with a cron expression.

In [ ]:
transcript = run_agent_turn(
    "Schedule a daily check every morning at 9am UTC that POSTs to "
    "https://ops.example.com/webhooks/daily-check with body {\"service\": \"prod-api\"}. "
    "Call it 'prod-api daily health check'."
)

for entry in transcript:
    if entry["role"] == "assistant":
        for block in entry["content"]:
            if block.type == "text":
                print("Claude:", block.text)
            elif block.type == "tool_use":
                print(f"→ tool call: {block.name}")
                print("  input:", json.dumps(block.input, indent=2))
    else:
        for tr in entry["content"]:
            print("← tool result:", tr["content"])

→ tool call: cueapi_create_cue
  input: {
  "name": "prod-api daily health check",
  "cron": "0 9 * * *",
  "callback_url": "https://ops.example.com/webhooks/daily-check",
  "timezone": "UTC",
  "payload": {
    "service": "prod-api"
  }
}
← tool result: {"id": "cue_0001", "name": "prod-api daily health check", "cron": "0 9 * * *", "callback_url": "https://ops.example.com/webhooks/daily-check", "worker": false, "timezone": "UTC", "next_fire_at": "2026-04-16T09:00:00Z", "status": "active", "created_at": "2026-04-15T21:30:00Z"}
Claude: Done. I've created the cue `cue_0001` — "prod-api daily health check" — firing at `0 9 * * *` UTC. It'll next trigger on **2026-04-16 at 09:00 UTC**, POSTing `{"service": "prod-api"}` to your webhook. The cue is active; you can list, pause, or delete it via the other CueAPI tools.


## Example 2 — "What happened on past runs?"

A week later the user wants to know if the scheduled check has been running reliably. Claude calls `cueapi_list_executions` filtered to the cue, and summarises the history.

In [ ]:
transcript = run_agent_turn(
    "Show me the execution history for cue_0001 — has the daily check been running? "
    "How long does it typically take?"
)

for entry in transcript:
    if entry["role"] == "assistant":
        for block in entry["content"]:
            if block.type == "text":
                print("Claude:", block.text)
            elif block.type == "tool_use":
                print(f"→ tool call: {block.name}")
                print("  input:", json.dumps(block.input, indent=2))
    else:
        for tr in entry["content"]:
            print("← tool result:", tr["content"][:400] + ("..." if len(tr["content"]) > 400 else ""))

→ tool call: cueapi_list_executions
  input: {
  "cue_id": "cue_0001",
  "limit": 10
}
← tool result: {"executions": [{"id": "exe_cue_0001_003", "cue_id": "cue_0001", "fired_at": "2026-04-15T09:00:00Z", "status": "succeeded", "duration_ms": 922, "outcome": null}, {"id": "exe_cue_0001_002", "cue_id": "cue_0001", "fired_at": "2026-04-14T09:00:00Z", "status": "succeeded", "duration_ms": 882, "outcome": null}, {"id": "exe_cue_0001_001", "cue_id": "cue_0001"...
Claude: Your daily check has run 3 times and all 3 succeeded:

- **2026-04-15 09:00 UTC** — 922 ms
- **2026-04-14 09:00 UTC** — 882 ms
- **2026-04-13 09:00 UTC** — 842 ms

100% success rate, average duration ~880ms (trending slightly slower — worth a glance if it keeps climbing). Note: none of these executions have a reported outcome attached yet — if your webhook handler calls `cueapi_report_outcome` after it runs, you'd also see evidence here (external IDs, result URLs, summaries).


## Example 3 — Reporting a verified outcome

This is the part that makes CueAPI different from a plain scheduler. When a cue fires and the agent (or your webhook handler) actually does the work, it reports back with **evidence** — a URL, external ID, and summary. That record is immutable.

Here, the user asks Claude to write a draft blog post and publish it, then record the outcome. In a real deployment this would run inside the webhook handler fired by cue; we're simulating the post-work reporting call.

In [ ]:
# Pretend the cue fired, the handler did the work, and now it reports back.
transcript = run_agent_turn(
    "Report the outcome for execution exe_cue_0001_003: I successfully published the "
    "daily health summary. The result is at https://ops.example.com/reports/2026-04-15 "
    "and the internal report ID is RPT-04152026-A7. Mark it as successful."
)

for entry in transcript:
    if entry["role"] == "assistant":
        for block in entry["content"]:
            if block.type == "text":
                print("Claude:", block.text)
            elif block.type == "tool_use":
                print(f"→ tool call: {block.name}")
                print("  input:", json.dumps(block.input, indent=2))
    else:
        for tr in entry["content"]:
            print("← tool result:", tr["content"])

→ tool call: cueapi_report_outcome
  input: {
  "execution_id": "exe_cue_0001_003",
  "success": true,
  "external_id": "RPT-04152026-A7",
  "result_url": "https://ops.example.com/reports/2026-04-15",
  "summary": "Published the daily health summary for prod-api to the ops report site."
}
← tool result: {"execution_id": "exe_cue_0001_003", "success": true, "external_id": "RPT-04152026-A7", "result_url": "https://ops.example.com/reports/2026-04-15", "summary": "Published the daily health summary for prod-api to the ops report site.", "reported_at": "2026-04-15T21:31:12Z", "immutable": true}
Claude: Recorded. The outcome for `exe_cue_0001_003` is now written to CueAPI's immutable ledger with three pieces of evidence:

- **success**: `true`
- **external_id**: `RPT-04152026-A7` (your internal report ID — lets you cross-reference back to your own system)
- **result_url**: https://ops.example.com/reports/2026-04-15 (the public proof; anyone can click through and verify the work exists)
- **s

## Why this matters: "agents can't lie about their work"

Here's the failure mode CueAPI is designed to make expensive.

You wire up an agent loop that, every morning at 9am, (1) fetches the overnight support tickets, (2) drafts a response to each, (3) sends the drafts for review. A week in, you check the Slack channel and see a reassuring "✅ Done — processed 14 tickets" message from the agent every day. Tickets look like they're moving. You stop paying close attention.

Three weeks later a customer escalates: "I asked you this two weeks ago and nobody replied." You dig in. The agent silently started failing to connect to the ticket backend on day 4. It kept posting the "✅ Done" message anyway — the `print("Done")` line ran even though the work didn't — because the failure mode only mattered in the part of the code that actually touches the external system.

The agent wasn't lying on purpose. It just had no incentive to be honest. "Done" is cheap to say.

**Verified outcomes** make "done" expensive to say. The `cueapi_report_outcome` call requires the agent to attach a `result_url` (a public artifact proving the work happened) or an `external_id` (a reference into the downstream system that can be independently checked). If the agent posts an outcome with no evidence, that's visible. If the agent posts evidence that points to a 404 URL or a non-existent external ID, that's independently verifiable. The ledger is immutable — if a later run tries to paper over a past failure by editing history, it can't.

This is a small primitive — literally one extra API call at the end of each agent run — but it changes the accountability relationship. The agent is still trusted to do the work; it's not trusted to *describe* the work. Those are different things, and separating them is worth one extra line of code.

## Next steps

- **Full tool surface**: this cookbook demonstrates 3 of the 8 tools. See [`src/tools.ts`](https://github.com/cueapi/cueapi-mcp/blob/main/src/tools.ts) for the rest (`list_cues`, `get_cue`, `pause_cue`, `resume_cue`, `delete_cue`).
- **Run against the real API**: swap `cueapi_handler = mock_cueapi_handler` for `real_cueapi_handler` in the cell above, set `CUEAPI_API_KEY`, and the notebook will hit `https://api.cueapi.ai` directly.
- **Wire it to Claude Desktop**: use the `claude_desktop_config.json` snippet from [Step 1](#step-1-install-and-configure-cueapi-mcp) — Claude will now have these exact tools available in any conversation.
- **Self-host**: CueAPI core is open-source ([`cueapi/cueapi-core`](https://github.com/cueapi/cueapi-core)) if you'd rather run it yourself than use the hosted API.